In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_fscore_support, confusion_matrix

In [2]:
print("="*60)
print("CREDIT SCORING MODEL - CODEALPHA MACHINE LEARNING INTERNSHIP")
print("="*60)

CREDIT SCORING MODEL - CODEALPHA MACHINE LEARNING INTERNSHIP


In [3]:
# ---------------------------------------------------------
# Step 1: Simulate / Load Dataset
# ---------------------------------------------------------
np.random.seed(42)
n_samples = 2000

print("\n[INFO] Generating simulated financial dataset...")
df = pd.DataFrame({
    'income': np.random.normal(55000, 18000, n_samples).clip(15000, 150000),
    'debts': np.random.normal(18000, 7000, n_samples).clip(1000, 80000),
    'payment_history_score': np.random.uniform(0.4, 1.0, n_samples),
    'credit_utilization': np.random.uniform(0.05, 0.95, n_samples),
    'recent_inquiries': np.random.poisson(lam=1.5, size=n_samples)
})

# Define target: Creditworthiness (1 = Creditworthy, 0 = High Risk / Default)
# Higher income and payment history boost creditworthiness; high debts and utilization lower it.
latent_score = (
    (df['income'] / 20000) 
    - (df['debts'] / 15000) 
    + (df['payment_history_score'] * 3) 
    - (df['credit_utilization'] * 2)
    - (df['recent_inquiries'] * 0.3)
    + np.random.normal(0, 0.8, n_samples)
)
probabilities = 1 / (1 + np.exp(-latent_score))
df['credit_worthiness'] = (probabilities > 0.5).astype(int)

print(f"[INFO] Dataset shape: {df.shape}")
print(f"[INFO] Class distribution:\n{df['credit_worthiness'].value_counts(normalize=True)}")


[INFO] Generating simulated financial dataset...
[INFO] Dataset shape: (2000, 6)
[INFO] Class distribution:
credit_worthiness
1    0.9295
0    0.0705
Name: proportion, dtype: float64


In [4]:
# ---------------------------------------------------------
# Step 2: Feature Engineering
# ---------------------------------------------------------
print("\n[INFO] Performing feature engineering...")
# Debt-to-Income (DTI) Ratio
df['dti_ratio'] = df['debts'] / (df['income'] + 1)
# Total Credit Health Index
df['credit_health_index'] = df['payment_history_score'] * (1 - df['credit_utilization'])

# Select features and target
features = ['income', 'debts', 'payment_history_score', 'credit_utilization', 'recent_inquiries', 'dti_ratio', 'credit_health_index']
X = df[features]
y = df['credit_worthiness']

# Train-Test Split (Stratified to preserve class imbalance ratios)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Feature Scaling (Crucial for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


[INFO] Performing feature engineering...


In [5]:
# ---------------------------------------------------------
# Step 3: Model Initialization & Training
# ---------------------------------------------------------
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42)
}

results = []

print("\n[INFO] Training models and evaluating performance...\n")
for name, model in models.items():
    # Use scaled data for Logistic Regression, raw data for Tree-based models
    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
    
    # Calculate Metrics
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary')
    roc_auc = roc_auc_score(y_test, y_prob)
    
    results.append({
        "Model": name,
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1-Score": round(f1, 4),
        "ROC-AUC": round(roc_auc, 4)
    })
    
    print(f"--- {name} Classification Report ---")
    print(classification_report(y_test, y_pred))


[INFO] Training models and evaluating performance...

--- Logistic Regression Classification Report ---
              precision    recall  f1-score   support

           0       0.65      0.39      0.49        28
           1       0.96      0.98      0.97       372

    accuracy                           0.94       400
   macro avg       0.80      0.69      0.73       400
weighted avg       0.93      0.94      0.94       400

--- Decision Tree Classification Report ---
              precision    recall  f1-score   support

           0       0.63      0.43      0.51        28
           1       0.96      0.98      0.97       372

    accuracy                           0.94       400
   macro avg       0.79      0.70      0.74       400
weighted avg       0.94      0.94      0.94       400

--- Random Forest Classification Report ---
              precision    recall  f1-score   support

           0       0.75      0.32      0.45        28
           1       0.95      0.99      0.97 

In [6]:
# ---------------------------------------------------------
# Step 4: Results Summary
# ---------------------------------------------------------
results_df = pd.DataFrame(results)
print("="*60)
print("FINAL PERFORMANCE BENCHMARK SUMMARY")
print("="*60)
print(results_df.to_string(index=False))
print("="*60)

FINAL PERFORMANCE BENCHMARK SUMMARY
              Model  Precision  Recall  F1-Score  ROC-AUC
Logistic Regression     0.9556  0.9839    0.9695   0.9362
      Decision Tree     0.9580  0.9812    0.9695   0.8705
      Random Forest     0.9510  0.9919    0.9711   0.9096
